In [1]:
from pathlib import Path
from collections import Counter
from urllib.parse import urlparse, parse_qs
from datetime import datetime, timezone
import json
import yt_dlp

1. Read video IDs or URLs from a file, one line at a time

In [2]:
def stream_video_urls(file_path):
    """Yield each non-empty, non-comment line from a text file."""

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            item = line.strip()

            # Ignore blank lines and comments.
            if item and not item.startswith("#"):
                yield item
    

In [3]:
video_urls = list(stream_video_urls("/home/ibukhari/Documents/Dev/Signet_PersonalExperiments/datasets/youtube-asl_youtube_asl_video_ids.txt"))

In [4]:
sample_index = 345
print(f"Total Videos: {len(video_urls)}")
print(f"Video URL at index {sample_index}:   {video_urls[sample_index]}")

Total Videos: 11096
Video URL at index 345:   N4TPKucBOyU


2. Extract the YouTube video ID from an input (extra check but not needed as txt file is clean)

In [5]:
def get_video_id(value):
    """Accept a YouTube video ID or a supported YouTube URL."""

    value = str(value).strip()

    if not value:
        return None

    # A plain ID does not contain a URL scheme or path.
    if "://" not in value and "/" not in value:
        return value

    parsed = urlparse(value)
    host = (parsed.hostname or "").lower()

    # Example: https://youtu.be/VIDEO_ID
    if host == "youtu.be":
        return parsed.path.strip("/").split("/")[0] or None

    # Examples: youtube.com/watch?v=VIDEO_ID
    #            youtube.com/shorts/VIDEO_ID
    #            youtube.com/embed/VIDEO_ID
    if host == "youtube.com" or host.endswith(".youtube.com"):
        if parsed.path == "/watch":
            return parse_qs(parsed.query).get("v", [None])[0]

        parts = parsed.path.strip("/").split("/")

        if len(parts) >= 2 and parts[0] in {
            "embed", "shorts", "live", "v"
        }:
            return parts[1]

    return None

3. Download Videos and Subs

In [ ]:
def download_youtube_videos(
    video_urls,
    video_output_path,
    subtitle_output_path,
    subtitle_langs=("en.*", "eng.*", "english.*"),
    write_auto_subs=False,
    verbose=False,
    report_path="download_report.jsonl",
):

    # Prepare output directories.
    video_dir = Path(video_output_path).expanduser().resolve()
    subtitle_dir = Path(subtitle_output_path).expanduser().resolve()
    report_file = Path(report_path).expanduser().resolve()

    video_dir.mkdir(parents=True, exist_ok=True)
    subtitle_dir.mkdir(parents=True, exist_ok=True)
    report_file.parent.mkdir(parents=True, exist_ok=True)

    # File extensions used to identify downloaded outputs.
    video_exts = {".mp4", ".mkv", ".webm", ".mov", ".avi"}
    subtitle_exts = {".srt", ".vtt", ".ass", ".lrc"}

    # Configure yt-dlp.
    options = {
        "format": (
            "bestvideo[ext=mp4]+bestaudio[ext=m4a]/"
            "best[ext=mp4]/bestvideo+bestaudio/best"
        ),
        "merge_output_format": "mp4",

        # Save videos and subtitles in separate directories.
        "outtmpl": {
            "default": str(video_dir / "%(id)s.%(ext)s"),
            "subtitle": str(subtitle_dir / "%(id)s.%(ext)s"),
        },
        "paths": {
            "home": str(video_dir),
            "subtitle": str(subtitle_dir),
        },

        # Subtitle settings.
        "writesubtitles": True,
        "writeautomaticsub": write_auto_subs,
        "subtitleslangs": list(subtitle_langs),
        # "subtitlesformat": "srt",

        # Download behaviour.
        "ffmpeg_location": "/usr/bin/ffmpeg",
        "ignoreerrors": False,
        "noplaylist": True,
        "quiet": not verbose,
        "no_warnings": not verbose,
        "noprogress": True,
    }

    # Append mode preserves results from previous runs.
    with report_file.open("a", encoding="utf-8") as report:

        # Reuse one yt-dlp instance for all inputs.
        with yt_dlp.YoutubeDL(options) as ydl:

            for item in video_urls:
                url = str(item).strip()

                # Ignore blank inputs and comments.
                if not url or url.startswith("#"):
                    continue

                video_id = get_video_id(url)

                # Every input gets its own report record.
                record = {
                    "input": url,
                    "video_id": video_id,
                    "timestamp": datetime.now(timezone.utc).isoformat(),
                    "status": "failed",
                    "video_files": [],
                    "subtitle_files": [],
                    "error": None,
                }

                try:
                    if not video_id:
                        raise ValueError(
                            "Invalid YouTube URL or video ID"
                        )

                    

                    # Ask yt-dlp to retrieve information and download.
                    info = ydl.extract_info(url, download=True)

                    if not info:
                        raise RuntimeError(
                            "No video information returned"
                        )

                    actual_id = info.get("id", video_id)

                    # Look for a video file matching this video's ID.
                    video_files = [
                        path
                        for path in video_dir.glob(f"{actual_id}.*")
                        if path.is_file()
                        and path.suffix.lower() in video_exts
                    ]

                    # Look for subtitle files matching this video's ID.
                    subtitle_files = [
                        path
                        for path in subtitle_dir.glob(f"{actual_id}.*")
                        if path.is_file()
                        and path.suffix.lower() in subtitle_exts
                    ]

                    # A video counts as available if an output file exists.
                    if not video_files:
                        raise RuntimeError(
                            "No video output file detected"
                        )

                    record.update({
                        "status": "success",
                        "video_files": [str(p) for p in video_files],
                        "subtitle_files": [str(p) for p in subtitle_files],
                    })

                except Exception as exc:
                    # Record the failure and continue to the next input.
                    record["error"] = str(exc)

                # Save this video's result immediately.
                report.write(
                    json.dumps(record, ensure_ascii=False) + "\n"
                )
                report.flush()

                if verbose:
                    print(
                        f"[{record['status'].upper()}] "
                        f"{video_id or url}"
                    )

    # Return the location of the report, not aggregate statistics.
    return str(report_file)



In [12]:

def download_youtube_videos(
    video_urls,
    video_output_path,
    subtitle_output_path,
    subtitle_langs=("en.*", "eng.*", "english.*"),
    write_auto_subs=False,
    verbose=False,
    report_path="download_report.jsonl",
):

    # Create the output directories and report path.
    video_dir = Path(video_output_path).expanduser().resolve()
    subtitle_dir = Path(subtitle_output_path).expanduser().resolve()
    report_file = Path(report_path).expanduser().resolve()

    video_dir.mkdir(parents=True, exist_ok=True)
    subtitle_dir.mkdir(parents=True, exist_ok=True)
    report_file.parent.mkdir(parents=True, exist_ok=True)

    # Define the file extensions to recognise as videos and subtitles.
    video_exts = {".mp4", ".mkv", ".webm", ".mov", ".avi"}
    subtitle_exts = {".srt", ".vtt", ".ass", ".lrc"}

    # Configure yt-dlp's video, subtitle, and output settings.
    options = {
            "format": (
                "bestvideo[ext=mp4]+bestaudio[ext=m4a]/"
                "best[ext=mp4]/bestvideo+bestaudio/best"
            ),
            "merge_output_format": "mp4",
    
            # Save videos and subtitles in separate directories.
            "outtmpl": {
                "default": str(video_dir / "%(id)s.%(ext)s"),
                "subtitle": str(subtitle_dir / "%(id)s.%(ext)s"),
            },
            "paths": {
                "home": str(video_dir),
                "subtitle": str(subtitle_dir),
            },
    
            # Subtitle settings.
            "writesubtitles": True,
            "writeautomaticsub": write_auto_subs,
            "subtitleslangs": list(subtitle_langs),
            # "subtitlesformat": "srt",
    
            # Download behaviour.
            "ffmpeg_location": "/usr/bin/ffmpeg",
            "ignoreerrors": False,
            "noplaylist": True,
            "quiet": not verbose,
            "no_warnings": not verbose,
            "noprogress": True,
    }

    # Open the report in append mode to preserve previous results.
    with report_file.open("a", encoding="utf-8") as report:

        # Create one yt-dlp instance and reuse it for all videos.
        with yt_dlp.YoutubeDL(options) as ydl:

            # Process each URL or video ID individually.
            for item in video_urls:

                url = str(item).strip()

                # Ignore empty lines and comments.
                if not url or url.startswith("#"):
                    continue

                # Extract the video ID from the input.
                video_id = get_video_id(url)

                # Initialise a report record for the current video.
                record = {
                    "input": url,
                    "video_id": video_id,
                    "timestamp": datetime.now(timezone.utc).isoformat(),
                    "status": "failed",
                    "video_files": [],
                    "subtitle_files": [],
                    "error": None,
                }

                try:
                    # Reject inputs that are not recognised as YouTube IDs or URLs.
                    if not video_id:
                        raise ValueError("Invalid YouTube URL or video ID")

                    # Check whether the video file already exists.
                    existing_videos = [
                        p for p in video_dir.glob(f"{video_id}.*")
                        if p.is_file()
                        and p.suffix.lower() in video_exts
                    ]

                    video_exists = bool(existing_videos)

                    # Check whether subtitle files already exist.
                    existing_subtitles = [
                        p for p in subtitle_dir.glob(f"{video_id}.*")
                        if p.is_file()
                        and p.suffix.lower() in subtitle_exts
                    ]

                    subtitles_exist = bool(existing_subtitles)

                    # Reuse existing files if both the video and subtitles exist.
                    if video_exists and subtitles_exist:

                        video_files = existing_videos
                        subtitle_files = existing_subtitles

                        if verbose:
                            print(f"[SKIP] Already downloaded: {video_id}")

                    else:

                        # Skip video downloading if the video already exists.
                        ydl.params["skip_download"] = video_exists

                        try:
                            # Download the missing video and/or subtitles.
                            info = ydl.extract_info(url, download=True)
                        finally:
                            # Reset skip_download before processing the next video.
                            ydl.params.pop("skip_download", None)

                        # Check whether yt-dlp returned video information.
                        if not info:
                            raise RuntimeError("No video information returned")

                        actual_id = info.get("id", video_id)

                        # Find the video file in the output directory.
                        video_files = [
                            p for p in video_dir.glob(f"{actual_id}.*")
                            if p.is_file()
                            and p.suffix.lower() in video_exts
                        ]

                        # Find the subtitle files in the subtitle directory.
                        subtitle_files = [
                            p for p in subtitle_dir.glob(f"{actual_id}.*")
                            if p.is_file()
                            and p.suffix.lower() in subtitle_exts
                        ]

                    # Ensure that a video file is available.
                    if not video_files:
                        raise RuntimeError("No video output file detected")

                    # Mark the result according to subtitle availability.
                    if subtitle_files:
                        record["status"] = "complete"
                    else:
                        record["status"] = "video_only"
                        record["error"] = "No matching subtitle file found"

                    # Add the discovered output files to the report.
                    record["video_files"] = [str(p) for p in video_files]
                    record["subtitle_files"] = [str(p) for p in subtitle_files]

                except Exception as exc:

                    # Record the error without stopping the remaining downloads.
                    record["status"] = "failed"
                    record["error"] = str(exc)

                # Write the current video's result as one JSONL record.
                report.write(
                    json.dumps(record, ensure_ascii=False) + "\n"
                )
                report.flush()

                # Print the outcome when verbose mode is enabled.
                if verbose:
                    print(
                        f"[{record['status'].upper()}] "
                        f"{video_id or url} | "
                        f"Subtitles: {len(record['subtitle_files'])}"
                    )

    # Return the report location for subsequent statistics tracking.
    return str(report_file)


4. Calculate statistics from the completed download report

In [7]:
def track_download_stats(report_path, verbose=True):
    """
    Read the JSONL report and calculate aggregate statistics.
    """

    report_file = Path(report_path).expanduser().resolve()

    # Counters store the aggregate results.
    stats = {
        "inputs_processed": 0,
        "videos_available": 0,
        "videos_failed": 0,
        "subtitles_found": 0,
        "video_formats": Counter(),
        "subtitle_formats": Counter(),
        "failure_reasons": Counter(),
    }

    # Read the report one line at a time.
    with report_file.open("r", encoding="utf-8") as report:

        for line in report:
            if not line.strip():
                continue

            record = json.loads(line)

            stats["inputs_processed"] += 1

            if record["status"] == "success":
                stats["videos_available"] += 1

                # Count the extensions of the video files in this record.
                for filename in record["video_files"]:
                    extension = Path(filename).suffix.lower()
                    stats["video_formats"][extension] += 1

            else:
                stats["videos_failed"] += 1

                # Count the reported failure reason.
                reason = record.get("error") or "Unknown error"
                stats["failure_reasons"][reason] += 1

            # Count subtitle files for successful records.
            for filename in record.get("subtitle_files", []):
                extension = Path(filename).suffix.lower()
                stats["subtitle_formats"][extension] += 1
                stats["subtitles_found"] += 1

    # Convert Counters into regular dictionaries for the result.
    stats["video_formats"] = dict(stats["video_formats"])
    stats["subtitle_formats"] = dict(stats["subtitle_formats"])
    stats["failure_reasons"] = dict(stats["failure_reasons"])
    stats["report_path"] = str(report_file)

    if verbose:
        print("\n=== Download Statistics ===")
        print(f"Inputs processed: {stats['inputs_processed']}")
        print(f"Videos available: {stats['videos_available']}")
        print(f"Failed videos: {stats['videos_failed']}")
        print(f"Subtitle files found: {stats['subtitles_found']}")
        print(f"Video formats: {stats['video_formats']}")
        print(f"Subtitle formats: {stats['subtitle_formats']}")
        print(f"Failure reasons: {stats['failure_reasons']}")
        print(f"Report: {report_file}")

    return stats



Run the workflow

In [13]:
# Run Downloads
report_path = download_youtube_videos(
    video_urls=video_urls[:10],
    video_output_path="datasets/Videos",
    subtitle_output_path="datasets/Subtitles",
    subtitle_langs=("en",),
    write_auto_subs=False,
    verbose=True,
    report_path="./download_report.jsonl",
)


[SKIP] Already downloaded: 4eNt91uV02o
[COMPLETE] 4eNt91uV02o | Subtitles: 1
[youtube] Extracting URL: UznY5SfH0RI
[youtube] UznY5SfH0RI: Downloading webpage


[youtube] UznY5SfH0RI: Downloading visionos player API JSON
[youtube] UznY5SfH0RI: Downloading m3u8 information
[info] UznY5SfH0RI: Downloading 1 format(s): 135+140


[info] There are no subtitles for the requested languages
[VIDEO_ONLY] UznY5SfH0RI | Subtitles: 0
[youtube] Extracting URL: JfWFKcT8wPQ
[youtube] JfWFKcT8wPQ: Downloading webpage


[youtube] JfWFKcT8wPQ: Downloading visionos player API JSON
[youtube] JfWFKcT8wPQ: Downloading m3u8 information
[info] JfWFKcT8wPQ: Downloading 1 format(s): 136+140
[info] There are no subtitles for the requested languages
[VIDEO_ONLY] JfWFKcT8wPQ | Subtitles: 0
[SKIP] Already downloaded: vOdoFInjH8s
[COMPLETE] vOdoFInjH8s | Subtitles: 1
[youtube] Extracting URL: eOJfTd3nybA
[youtube] eOJfTd3nybA: Downloading webpage


[youtube] eOJfTd3nybA: Downloading visionos player API JSON


...

ERROR: [youtube] LSmWzwmlnxc: This video is not available


[FAILED] LSmWzwmlnxc | Subtitles: 0
[youtube] Extracting URL: SqxjhX6wunw
[youtube] SqxjhX6wunw: Downloading webpage


[youtube] SqxjhX6wunw: Downloading visionos player API JSON
[youtube] SqxjhX6wunw: Downloading m3u8 information
[info] SqxjhX6wunw: Downloading 1 format(s): 614+140
[info] There are no subtitles for the requested languages
[VIDEO_ONLY] SqxjhX6wunw | Subtitles: 0
[SKIP] Already downloaded: _FW1JoDAW8w
[COMPLETE] _FW1JoDAW8w | Subtitles: 1
[youtube] Extracting URL: oY4zNmRl79Q
[youtube] oY4zNmRl79Q: Downloading webpage


[youtube] oY4zNmRl79Q: Downloading visionos player API JSON
[youtube] oY4zNmRl79Q: Downloading m3u8 information
[info] oY4zNmRl79Q: Downloading 1 format(s): 137+140
[info] There are no subtitles for the requested languages
[VIDEO_ONLY] oY4zNmRl79Q | Subtitles: 0
[youtube] Extracting URL: ErnACzXrYAA
[youtube] ErnACzXrYAA: Downloading webpage


[youtube] ErnACzXrYAA: Downloading visionos player API JSON
[youtube] ErnACzXrYAA: Downloading m3u8 information
[info] ErnACzXrYAA: Downloading 1 format(s): 135+140
[info] There are no subtitles for the requested languages
[VIDEO_ONLY] ErnACzXrYAA | Subtitles: 0


In [14]:
# Run Stats
stats = track_download_stats(report_path)

print("\nReturned statistics:")
print(stats)


=== Download Statistics ===
Inputs processed: 28
Videos available: 14
Failed videos: 14
Subtitle files found: 9
Video formats: {'.mp4': 14}
Subtitle formats: {'.srt': 9}
Failure reasons: {'ERROR: [youtube] eOJfTd3nybA: This video is unavailable': 3, 'ERROR: [youtube] LSmWzwmlnxc: This video is not available': 3, 'Unknown error': 3, 'No matching subtitle file found': 5}
Report: /home/ibukhari/Documents/Dev/Signet_PersonalExperiments/download_report.jsonl

Returned statistics:
{'inputs_processed': 28, 'videos_available': 14, 'videos_failed': 14, 'subtitles_found': 9, 'video_formats': {'.mp4': 14}, 'subtitle_formats': {'.srt': 9}, 'failure_reasons': {'ERROR: [youtube] eOJfTd3nybA: This video is unavailable': 3, 'ERROR: [youtube] LSmWzwmlnxc: This video is not available': 3, 'Unknown error': 3, 'No matching subtitle file found': 5}, 'report_path': '/home/ibukhari/Documents/Dev/Signet_PersonalExperiments/download_report.jsonl'}
